# Final Summary Grid Search (T+5 / T+3)

This notebook reuses the current project pipeline logic and evaluates two horizons separately:
- Cell for T+5
- Cell for T+3

Methods tried inside each horizon cell:
1. LGBM rank model on switching labels (2/1/0)
2. MLP rank model on switching labels (2/1/0)
3. LGBM regression on 30-day truncated remaining days
4. LGBM regression on T+n vshare
5. Three-model ensemble

In [1]:
import ast
import itertools
import types
import warnings

import chinese_calendar
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Keep notebook independent from Streamlit runtime by loading only required symbols from main.py.
def _load_main_runtime(main_path="main.py"):
    with open(main_path, "r", encoding="utf-8") as f:
        src = f.read()

    mod = ast.parse(src, filename=main_path)

    keep_funcs = {
        "define_main_adaptive_robust",
        "make_refined_features",
        "get_training_samples_aggressive",
    }
    keep_consts = {
        "RANDOM_SEED",
        "PARQUET_FILE",
        "FEATURE_COLS",
        "EXCLUDED_SYMBOLS",
    }

    body = []
    for node in mod.body:
        if isinstance(node, ast.FunctionDef) and node.name in keep_funcs:
            body.append(node)
            continue

        if isinstance(node, ast.Assign):
            names = []
            for t in node.targets:
                if isinstance(t, ast.Name):
                    names.append(t.id)
            if any(name in keep_consts for name in names):
                body.append(node)
            continue

        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
            if node.target.id in keep_consts:
                body.append(node)

    runtime_mod = ast.Module(body=body, type_ignores=[])
    ast.fix_missing_locations(runtime_mod)

    ns = {
        "__builtins__": __builtins__,
        "np": np,
        "pd": pd,
        "chinese_calendar": chinese_calendar,
    }

    exec(compile(runtime_mod, main_path, "exec"), ns)
    return ns

_main_ns = _load_main_runtime("main.py")
m = types.SimpleNamespace(**_main_ns)

RANDOM_SEED = int(m.RANDOM_SEED)
PARQUET_FILE = m.PARQUET_FILE
FEATURE_COLS = list(m.FEATURE_COLS)
TEST_START_DATE = "2025-11-01"
TRAIN_HISTORY_DAYS = 500
MAX_TRIALS_PER_HORIZON = 160

# New: search over alternative main-definition rules.
MAIN_RULE_GRID = [
    "adaptive",
    "oi_first_vol_top2",
    "volume_first",
]

SAMPLE_GRID_RANK = [
    {"window_before": 6, "window_after": 6, "noise_frac": 0.05},
    {"window_before": 8, "window_after": 8, "noise_frac": 0.10},
    {"window_before": 12, "window_after": 10, "noise_frac": 0.10},
]
SAMPLE_GRID_VSHARE = [
    {"window_before": 18, "window_after": 18, "noise_frac": 0.0},
    {"window_before": 20, "window_after": 20, "noise_frac": 0.0},
    {"window_before": 25, "window_after": 20, "noise_frac": 0.0},
]
SAMPLE_GRID_DAYS = [
    {"window_before": 15, "window_after": 15, "noise_frac": 0.0},
    {"window_before": 20, "window_after": 20, "noise_frac": 0.0},
    {"window_before": 30, "window_after": 20, "noise_frac": 0.0},
]

RANK_GRID = [
    {"learning_rate": 0.02, "num_leaves": 63, "feature_fraction": 0.85, "min_data_in_leaf": 20, "num_boost_round": 900},
    {"learning_rate": 0.02, "num_leaves": 127, "feature_fraction": 0.85, "min_data_in_leaf": 20, "num_boost_round": 900},
    {"learning_rate": 0.015, "num_leaves": 127, "feature_fraction": 0.70, "min_data_in_leaf": 20, "num_boost_round": 1100},
    {"learning_rate": 0.01, "num_leaves": 63, "feature_fraction": 0.60, "min_data_in_leaf": 20, "num_boost_round": 900},
    {"learning_rate": 0.01, "num_leaves": 255, "feature_fraction": 0.60, "min_data_in_leaf": 30, "num_boost_round": 1400},
]
VSHARE_GRID = [
    {"learning_rate": 0.02, "num_leaves": 63, "feature_fraction": 0.85, "min_data_in_leaf": 20, "num_boost_round": 900},
    {"learning_rate": 0.03, "num_leaves": 63, "feature_fraction": 0.70, "min_data_in_leaf": 20, "num_boost_round": 900},
    {"learning_rate": 0.01, "num_leaves": 127, "feature_fraction": 0.80, "min_data_in_leaf": 30, "num_boost_round": 1200},
]
DAYS_GRID = [
    {"learning_rate": 0.03, "num_leaves": 63, "feature_fraction": 0.85, "min_data_in_leaf": 20, "num_boost_round": 900},
    {"learning_rate": 0.02, "num_leaves": 63, "feature_fraction": 0.85, "min_data_in_leaf": 20, "num_boost_round": 900},
    {"learning_rate": 0.01, "num_leaves": 127, "feature_fraction": 0.80, "min_data_in_leaf": 30, "num_boost_round": 1200},
]

ENSEMBLE_GRID = [
    {"method": "additive", "w_rank": 1.0, "w_days": 0.0, "w_vshare": 0.0},
    {"method": "additive", "w_rank": 0.0, "w_days": 1.0, "w_vshare": 0.0},
    {"method": "additive", "w_rank": 0.0, "w_days": 0.0, "w_vshare": 1.0},
    {"method": "additive", "w_rank": 1.0, "w_days": 0.1, "w_vshare": 2.2},
    {"method": "additive", "w_rank": 1.0, "w_days": 0.2, "w_vshare": 2.5},
    {"method": "additive", "w_rank": 1.2, "w_days": 0.0, "w_vshare": 1.8},
    {"method": "exponential", "decay": 0.03, "gamma": 1.5},
    {"method": "exponential", "decay": 0.05, "gamma": 2.0},
    {"method": "multiplicative", "alpha": 7.0, "beta": 1.2, "base_pow": 1.0},
]


def min_max_scale(x):
    margin = x.max() - x.min()
    return (x - x.min()) / margin if margin > 1e-9 else x * 0.0


def _define_main_by_rule(df, main_rule):
    """
    main_rule:
      - adaptive: keep existing production logic
      - oi_first_vol_top2: oi rank1 and volume rank<=2 preferred
      - volume_first: volume rank1 only
    """
    if main_rule == "adaptive":
        return m.define_main_adaptive_robust(df)

    out = df.sort_values(["symbol_code", "date_dt", "delivery_code"]).copy()
    out["volume"] = out["volume"].replace(0, np.nan)
    out["oi"] = out["oi"].replace(0, np.nan)

    v_ranks = out.groupby(["symbol_code", "date_dt"])["volume"].rank(ascending=False, method="first", na_option="bottom")
    o_ranks = out.groupby(["symbol_code", "date_dt"])["oi"].rank(ascending=False, method="first", na_option="bottom")

    if main_rule == "oi_first_vol_top2":
        c1 = out["oi"].notna() & (o_ranks == 1) & out["volume"].notna() & (v_ranks <= 2)
        c2 = out["oi"].notna() & (o_ranks == 1)
    elif main_rule == "volume_first":
        c1 = out["volume"].notna() & (v_ranks == 1)
        c2 = out["volume"].notna() & (v_ranks <= 2)
    else:
        raise ValueError(f"Unknown main_rule: {main_rule}")

    out["tmp_score"] = np.where(c1, 2, np.where(c2, 1, 0))
    max_score = out.groupby(["symbol_code", "date_dt"])["tmp_score"].transform("max")

    out["is_candidate"] = (out["tmp_score"] == max_score) & (out["tmp_score"] > 0)
    candidates = out[out["is_candidate"]][["symbol_code", "date_dt", "delivery_code"]].drop_duplicates(subset=["symbol_code", "date_dt"])
    candidates = candidates.rename(columns={"delivery_code": "main_code_cand"})

    sym_dates = out[["symbol_code", "date_dt"]].drop_duplicates().sort_values(["symbol_code", "date_dt"])
    sym_dates = sym_dates.merge(candidates, on=["symbol_code", "date_dt"], how="left")
    sym_dates["main_code_final"] = sym_dates.groupby("symbol_code")["main_code_cand"].ffill()

    out = out.merge(sym_dates[["symbol_code", "date_dt", "main_code_final"]], on=["symbol_code", "date_dt"], how="left")

    fallback = out[v_ranks == 1][["symbol_code", "date_dt", "delivery_code"]].drop_duplicates(subset=["symbol_code", "date_dt"])
    fallback = fallback.rename(columns={"delivery_code": "fallback_main"})
    out = out.merge(fallback, on=["symbol_code", "date_dt"], how="left")
    out["main_code_final"] = out["main_code_final"].fillna(out["fallback_main"])

    out["daily_rank"] = np.where(out["delivery_code"] == out["main_code_final"], 1, 99)
    out["v_rank"] = v_ranks

    out.rename(columns={"main_code_final": "actual_main_code"}, inplace=True)
    out.drop(columns=["tmp_score", "is_candidate", "fallback_main"], inplace=True)
    return out


def build_final_eval_for_horizon(pred_df, ens_cfg, horizon):
    target_col = f"is_t{horizon}_main"
    tmp = pred_df.copy()
    tmp = tmp.sort_values(["date_dt", "symbol_code", "delivery_code"]).drop_duplicates(["date_dt", "symbol_code", "delivery_code"], keep="last")
    tmp["norm_rank"] = tmp.groupby(["date_dt", "symbol_code"])["pred_rank_raw"].transform(min_max_scale)

    method = ens_cfg.get("method", "multiplicative")
    if method == "multiplicative":
        base_pow = ens_cfg.get("base_pow", 1.0)
        alpha = ens_cfg.get("alpha", 5.0)
        beta = ens_cfg.get("beta", 1.0)
        rank_score = tmp["norm_rank"] ** base_pow
        inf_days = alpha / (np.clip(tmp["pred_days"], 0, None) + alpha)
        inf_vshare = np.clip(tmp["pred_vshare"], 0, None) * beta
        tmp["ensemble_score"] = rank_score * (1.0 + inf_days + inf_vshare)
    elif method == "additive":
        w_rank = ens_cfg.get("w_rank", 1.0)
        w_days = ens_cfg.get("w_days", 1.0)
        w_vshare = ens_cfg.get("w_vshare", 1.0)
        days_score = 1.0 / (np.clip(tmp["pred_days"], 0, None) + 1.0)
        tmp["ensemble_score"] = w_rank * tmp["norm_rank"] + w_days * days_score + w_vshare * np.clip(tmp["pred_vshare"], 0, None)
    else:
        decay = ens_cfg.get("decay", 0.1)
        gamma = ens_cfg.get("gamma", 1.0)
        day_penalty = np.exp(-decay * np.clip(tmp["pred_days"], 0, None))
        vshare_boost = np.clip(tmp["pred_vshare"], 0.001, None) ** gamma
        tmp["ensemble_score"] = tmp["norm_rank"] * day_penalty * vshare_boost

    tmp["ensemble_pred_rank"] = tmp.groupby(["date_dt", "symbol_code"])["ensemble_score"].rank(ascending=False, method="first")
    min_date = tmp["date_dt"].min() + pd.Timedelta(days=5)
    max_date = tmp["date_dt"].max() - pd.Timedelta(days=5)
    final_eval = tmp[(tmp["date_dt"] >= min_date) & (tmp["date_dt"] <= max_date)].copy()

    main_map = (
        tmp[tmp["daily_rank"] == 1]
        .sort_values(["date_dt", "symbol_code", "delivery_code"])
        .drop_duplicates(["date_dt", "symbol_code"], keep="first")
        .set_index(["date_dt", "symbol_code"])["delivery_code"]
    )
    final_eval["actual_main_code"] = final_eval.set_index(["date_dt", "symbol_code"]).index.map(main_map)

    liq_mask = final_eval["volume"].fillna(0).gt(0) | final_eval["oi"].fillna(0).gt(0)
    has_main_ref = final_eval["actual_main_code"].notna()
    final_eval["is_action"] = (final_eval["ensemble_pred_rank"] == 1) & has_main_ref & (final_eval["delivery_code"] != final_eval["actual_main_code"]) & liq_mask
    final_eval["is_real_switch_target"] = has_main_ref & final_eval[target_col] & (final_eval["delivery_code"] != final_eval["actual_main_code"])
    return final_eval


def evaluate_trial_like_cell3(final_eval):
    y_true = final_eval["is_real_switch_target"].astype(int)
    y_pred = final_eval["is_action"].astype(int)
    out = {
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    }
    switch_groups = final_eval[final_eval["is_real_switch_target"] == 1].copy()
    switch_groups = switch_groups.sort_values(["symbol_code", "delivery_code", "date_dt"])
    switch_groups["signal_day"] = switch_groups.groupby(["symbol_code", "delivery_code"]).cumcount() + 1
    for d in range(1, 6):
        day_df = switch_groups[switch_groups["signal_day"] == d]
        out[f"day{d}_hit_rate"] = float(day_df["is_action"].mean()) if len(day_df) > 0 else 0.0
        out[f"day{d}_count"] = int(len(day_df))
    return out


def _prepare_data_and_targets_horizon(path, horizon, main_rule="adaptive"):
    df = pd.read_parquet(path)
    df["date_dt"] = pd.to_datetime(df["date"])
    df["is_excluded"] = df["symbol_code"].isin(m.EXCLUDED_SYMBOLS)

    df = _define_main_by_rule(df, main_rule)

    mains = df[df["daily_rank"] == 1][["symbol_code", "date_dt", "delivery_code"]].sort_values(["symbol_code", "date_dt"])

    future_col = f"future_main_t{horizon}"
    is_col = f"is_t{horizon}_main"
    not_main_col = f"not_main_t0_t{horizon-1}"
    vshare_col = f"v_share_t{horizon}"

    mains[future_col] = mains.groupby("symbol_code")["delivery_code"].shift(-horizon)
    df = pd.merge(df, mains[["symbol_code", "date_dt", future_col]], on=["symbol_code", "date_dt"], how="left")

    main_seq = mains[["symbol_code", "date_dt", "delivery_code"]].rename(columns={"delivery_code": "main_t0"})
    for k in range(1, horizon):
        main_seq[f"main_t{k}"] = main_seq.groupby("symbol_code")["main_t0"].shift(-k)
    df = df.merge(main_seq, on=["symbol_code", "date_dt"], how="left")

    is_tn_main = df["delivery_code"] == df[future_col]
    cols_window = ["main_t0"] + [f"main_t{k}" for k in range(1, horizon)]
    has_full = df[cols_window].notna().all(axis=1)
    neq_all = np.ones(len(df), dtype=bool)
    for c in cols_window:
        neq_all &= (df["delivery_code"] != df[c])
    df[is_col] = is_tn_main
    df[not_main_col] = has_full & neq_all
    df = df.drop(columns=cols_window)

    df = m.make_refined_features(df)

    LABEL_SWITCH = 2
    LABEL_MAIN = 1

    df_sorted = df.sort_values(["symbol_code", "delivery_code", "date_dt"])
    df[vshare_col] = df_sorted.groupby(["symbol_code", "delivery_code"])["v_share"].shift(-horizon)

    unique_dates = np.sort(df["date_dt"].unique())
    date_to_idx = {d: i for i, d in enumerate(unique_dates)}
    df["date_idx"] = df["date_dt"].map(date_to_idx)

    mains_first = df[df["daily_rank"] == 1].groupby(["symbol_code", "delivery_code"])["date_idx"].min().reset_index()
    mains_first.rename(columns={"date_idx": "first_main_idx"}, inplace=True)
    df = df.merge(mains_first, on=["symbol_code", "delivery_code"], how="left")

    df["remaining_days"] = df["first_main_idx"] - df["date_idx"]
    df.loc[df["remaining_days"] < 0, "remaining_days"] = 0
    df["remaining_days"] = df["remaining_days"].fillna(30)
    df.loc[df["remaining_days"] > 30, "remaining_days"] = 30

    df["target_rank"] = np.where(df[is_col] & df[not_main_col], LABEL_SWITCH, np.where(df[is_col], LABEL_MAIN, 0)).astype(np.int32)
    df["target_vshare"] = df[vshare_col].fillna(0).astype(np.float32)

    # New rule requested: current main contract has 30 days target.
    df["target_days"] = df["remaining_days"].astype(np.float32)
    df.loc[df["daily_rank"] == 1, "target_days"] = 30.0

    return df.sort_index()


def _train_lgb_rank(train_df, eval_df, feat_cols_rank, cfg):
    q_train = train_df.groupby(["date_dt", "symbol_code"]).size().values
    q_eval = eval_df.groupby(["date_dt", "symbol_code"]).size().values
    ds_train = lgb.Dataset(train_df[feat_cols_rank], label=train_df["target_rank"], group=q_train, categorical_feature=["symbol_cat"])
    ds_eval = lgb.Dataset(eval_df[feat_cols_rank], label=eval_df["target_rank"], group=q_eval, categorical_feature=["symbol_cat"])
    model = lgb.train({"objective": "lambdarank", "metric": "ndcg", "ndcg_at": [1, 2], "seed": RANDOM_SEED, "verbose": -1, "n_jobs": 1, **cfg}, ds_train, valid_sets=[ds_eval], num_boost_round=cfg["num_boost_round"], callbacks=[lgb.log_evaluation(0)])
    return model


def _train_lgb_reg(train_df, eval_df, feat_cols_reg, target_col, cfg):
    ds_train = lgb.Dataset(train_df[feat_cols_reg], label=train_df[target_col])
    ds_eval = lgb.Dataset(eval_df[feat_cols_reg], label=eval_df[target_col])
    model = lgb.train({"objective": "regression", "metric": "rmse", "seed": RANDOM_SEED, "verbose": -1, "n_jobs": 1, **cfg}, ds_train, valid_sets=[ds_eval], num_boost_round=cfg["num_boost_round"], callbacks=[lgb.log_evaluation(0)])
    return model


def _train_mlp_rank(train_df, feat_cols_mlp, cfg):
    x = train_df[feat_cols_mlp].copy()
    x = x.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    y = train_df["target_rank"].astype(float).values
    model = MLPRegressor(
        hidden_layer_sizes=cfg["hidden_layer_sizes"],
        alpha=cfg["alpha"],
        learning_rate_init=cfg["learning_rate_init"],
        max_iter=cfg["max_iter"],
        random_state=RANDOM_SEED,
    )
    model.fit(x_scaled, y)
    return model, scaler


MLP_GRID = [
    {"hidden_layer_sizes": (64, 32), "alpha": 1e-4, "learning_rate_init": 1e-3, "max_iter": 1200},
    {"hidden_layer_sizes": (128, 64), "alpha": 5e-4, "learning_rate_init": 5e-4, "max_iter": 1500},
]


def run_horizon_experiment(horizon):
    prepared = {}
    for main_rule in MAIN_RULE_GRID:
        df = _prepare_data_and_targets_horizon(PARQUET_FILE, horizon, main_rule=main_rule)

        feat_cols_all = [c for c in FEATURE_COLS if c in df.columns]
        feat_cols_rank = [c for c in feat_cols_all if c != "symbol_cat"] + ["symbol_cat"]
        feat_cols_mlp = [c for c in feat_cols_all if c != "symbol_cat"]
        feat_cols_days = [c for c in feat_cols_all if c != "symbol_cat"]
        feat_cols_vshare = [c for c in feat_cols_all if c not in {"symbol_cat", "v_share"}]

        test_start = pd.to_datetime(TEST_START_DATE)
        train_start = test_start - pd.Timedelta(days=TRAIN_HISTORY_DAYS)

        target_future_col = f"future_main_t{horizon}"
        train_pool = df[(~df["is_excluded"]) & (df[target_future_col].notna())].copy()
        raw_train = train_pool[(train_pool["date_dt"] >= train_start) & (train_pool["date_dt"] < test_start)].copy()
        eval_test_base = train_pool[train_pool["date_dt"] >= test_start].copy()

        all_pool = df[df[target_future_col].notna()].copy()
        eval_test_all = all_pool[all_pool["date_dt"] >= test_start].copy()

        prepared[main_rule] = {
            "raw_train": raw_train,
            "eval_test_base": eval_test_base,
            "eval_test_all": eval_test_all,
            "feat_cols_rank": feat_cols_rank,
            "feat_cols_mlp": feat_cols_mlp,
            "feat_cols_days": feat_cols_days,
            "feat_cols_vshare": feat_cols_vshare,
        }

    print(f"[T+{horizon}] Rule: train uses non-excluded only; eval reports both non-excluded and all symbols.")
    print(f"[T+{horizon}] Rule: signal suppressed when both volume and oi are zero.")
    print(f"[T+{horizon}] Feature construction: define_main_adaptive_robust + make_refined_features (from main.py).")
    print(f"[T+{horizon}] Feature rule: lambdarank includes symbol_cat; other models exclude symbol_cat; vshare model excludes v_share.")
    print(f"[T+{horizon}] Main-rule grid: {MAIN_RULE_GRID}")

    rank_specs = []
    mlp_specs = []
    days_specs = []
    vshare_specs = []
    three_specs = []

    for main_rule, s_cfg, rank_cfg in itertools.product(MAIN_RULE_GRID, SAMPLE_GRID_RANK, RANK_GRID):
        rank_specs.append({"main_rule": main_rule, "family": "lgbm_rank", "s_rank": s_cfg, "rank_cfg": rank_cfg, "ens_cfg": ENSEMBLE_GRID[0]})

    for main_rule, s_cfg, mlp_cfg in itertools.product(MAIN_RULE_GRID, SAMPLE_GRID_RANK, MLP_GRID):
        mlp_specs.append({"main_rule": main_rule, "family": "mlp_rank", "s_rank": s_cfg, "mlp_cfg": mlp_cfg, "ens_cfg": ENSEMBLE_GRID[0]})

    for main_rule, s_cfg, days_cfg in itertools.product(MAIN_RULE_GRID, SAMPLE_GRID_DAYS, DAYS_GRID):
        days_specs.append({"main_rule": main_rule, "family": "days_reg", "s_days": s_cfg, "days_cfg": days_cfg, "ens_cfg": ENSEMBLE_GRID[1]})

    for main_rule, s_cfg, v_cfg in itertools.product(MAIN_RULE_GRID, SAMPLE_GRID_VSHARE, VSHARE_GRID):
        vshare_specs.append({"main_rule": main_rule, "family": "vshare_reg", "s_vshare": s_cfg, "v_cfg": v_cfg, "ens_cfg": ENSEMBLE_GRID[2]})

    for main_rule, s_rank, s_v, s_d, rank_cfg, v_cfg, d_cfg, ens_cfg in itertools.product(
        MAIN_RULE_GRID,
        SAMPLE_GRID_RANK,
        SAMPLE_GRID_VSHARE,
        SAMPLE_GRID_DAYS,
        RANK_GRID,
        VSHARE_GRID,
        DAYS_GRID,
        ENSEMBLE_GRID[3:],
    ):
        three_specs.append({
            "main_rule": main_rule,
            "family": "three_model",
            "s_rank": s_rank,
            "s_vshare": s_v,
            "s_days": s_d,
            "rank_cfg": rank_cfg,
            "v_cfg": v_cfg,
            "d_cfg": d_cfg,
            "ens_cfg": ens_cfg,
        })

    non_three = rank_specs + mlp_specs + days_specs + vshare_specs
    max_three = max(0, MAX_TRIALS_PER_HORIZON - len(non_three))
    trial_specs = non_three + three_specs[:max_three]

    rows = []
    best = None
    best_key = (-1.0, -1.0, -1.0)

    for i, spec in enumerate(trial_specs, start=1):
        family = spec["family"]
        main_rule = spec["main_rule"]
        prep = prepared[main_rule]
        try:
            raw_train = prep["raw_train"]
            eval_test_base = prep["eval_test_base"]
            eval_test_all = prep["eval_test_all"]
            feat_cols_rank = prep["feat_cols_rank"]
            feat_cols_mlp = prep["feat_cols_mlp"]
            feat_cols_days = prep["feat_cols_days"]
            feat_cols_vshare = prep["feat_cols_vshare"]

            pred_base = eval_test_base.copy()
            pred_all = eval_test_all.copy()

            if family == "lgbm_rank":
                tr_rank = m.get_training_samples_aggressive(raw_train, **spec["s_rank"], random_state=RANDOM_SEED)
                rank_model = _train_lgb_rank(tr_rank, eval_test_base, feat_cols_rank, spec["rank_cfg"])
                pred_base["pred_rank_raw"] = rank_model.predict(pred_base[feat_cols_rank])
                pred_all["pred_rank_raw"] = rank_model.predict(pred_all[feat_cols_rank])
                pred_base["pred_vshare"] = 0.0
                pred_all["pred_vshare"] = 0.0
                pred_base["pred_days"] = 0.0
                pred_all["pred_days"] = 0.0

            elif family == "mlp_rank":
                tr_rank = m.get_training_samples_aggressive(raw_train, **spec["s_rank"], random_state=RANDOM_SEED)
                mlp_model, mlp_scaler = _train_mlp_rank(tr_rank, feat_cols_mlp, spec["mlp_cfg"])
                x_base = pred_base[feat_cols_mlp].replace([np.inf, -np.inf], np.nan).fillna(0.0)
                x_all = pred_all[feat_cols_mlp].replace([np.inf, -np.inf], np.nan).fillna(0.0)
                pred_base["pred_rank_raw"] = mlp_model.predict(mlp_scaler.transform(x_base))
                pred_all["pred_rank_raw"] = mlp_model.predict(mlp_scaler.transform(x_all))
                pred_base["pred_vshare"] = 0.0
                pred_all["pred_vshare"] = 0.0
                pred_base["pred_days"] = 0.0
                pred_all["pred_days"] = 0.0

            elif family == "days_reg":
                tr_days = m.get_training_samples_aggressive(raw_train, **spec["s_days"], random_state=RANDOM_SEED)
                d_model = _train_lgb_reg(tr_days, eval_test_base, feat_cols_days, "target_days", spec["days_cfg"])
                pred_base["pred_days"] = d_model.predict(pred_base[feat_cols_days])
                pred_all["pred_days"] = d_model.predict(pred_all[feat_cols_days])
                pred_base["pred_vshare"] = 0.0
                pred_all["pred_vshare"] = 0.0
                pred_base["pred_rank_raw"] = -pred_base["pred_days"]
                pred_all["pred_rank_raw"] = -pred_all["pred_days"]

            elif family == "vshare_reg":
                tr_v = m.get_training_samples_aggressive(raw_train, **spec["s_vshare"], random_state=RANDOM_SEED)
                v_model = _train_lgb_reg(tr_v, eval_test_base, feat_cols_vshare, "target_vshare", spec["v_cfg"])
                pred_base["pred_vshare"] = v_model.predict(pred_base[feat_cols_vshare])
                pred_all["pred_vshare"] = v_model.predict(pred_all[feat_cols_vshare])
                pred_base["pred_days"] = 0.0
                pred_all["pred_days"] = 0.0
                pred_base["pred_rank_raw"] = pred_base["pred_vshare"]
                pred_all["pred_rank_raw"] = pred_all["pred_vshare"]

            else:
                tr_rank = m.get_training_samples_aggressive(raw_train, **spec["s_rank"], random_state=RANDOM_SEED)
                tr_v = m.get_training_samples_aggressive(raw_train, **spec["s_vshare"], random_state=RANDOM_SEED)
                tr_d = m.get_training_samples_aggressive(raw_train, **spec["s_days"], random_state=RANDOM_SEED)
                rank_model = _train_lgb_rank(tr_rank, eval_test_base, feat_cols_rank, spec["rank_cfg"])
                v_model = _train_lgb_reg(tr_v, eval_test_base, feat_cols_vshare, "target_vshare", spec["v_cfg"])
                d_model = _train_lgb_reg(tr_d, eval_test_base, feat_cols_days, "target_days", spec["d_cfg"])
                pred_base["pred_rank_raw"] = rank_model.predict(pred_base[feat_cols_rank])
                pred_all["pred_rank_raw"] = rank_model.predict(pred_all[feat_cols_rank])
                pred_base["pred_vshare"] = v_model.predict(pred_base[feat_cols_vshare])
                pred_all["pred_vshare"] = v_model.predict(pred_all[feat_cols_vshare])
                pred_base["pred_days"] = d_model.predict(pred_base[feat_cols_days])
                pred_all["pred_days"] = d_model.predict(pred_all[feat_cols_days])

            final_base = build_final_eval_for_horizon(pred_base, spec["ens_cfg"], horizon)
            final_all = build_final_eval_for_horizon(pred_all, spec["ens_cfg"], horizon)
            metrics_base = evaluate_trial_like_cell3(final_base)
            metrics_all = evaluate_trial_like_cell3(final_all)

            row = {
                "trial": i,
                "main_rule": main_rule,
                "family": family,
                "f1": metrics_base["f1"],
                "precision": metrics_base["precision"],
                "recall": metrics_base["recall"],
                "d1_hit": metrics_base["day1_hit_rate"],
                "d2_hit": metrics_base["day2_hit_rate"],
                "d3_hit": metrics_base["day3_hit_rate"],
                "d4_hit": metrics_base["day4_hit_rate"],
                "d5_hit": metrics_base["day5_hit_rate"],
                "all_f1": metrics_all["f1"],
                "all_precision": metrics_all["precision"],
                "all_recall": metrics_all["recall"],
                "spec": spec,
            }
            rows.append(row)

            key = (metrics_base["f1"], metrics_base["day3_hit_rate"], metrics_base["recall"])
            if key > best_key:
                best_key = key
                best = {
                    "trial": i,
                    "main_rule": main_rule,
                    "family": family,
                    "spec": spec,
                    "metrics_excluded": metrics_base,
                    "metrics_all": metrics_all,
                    "final_eval_excluded": final_base,
                    "final_eval_all": final_all,
                }

            print(f"[T+{horizon}] Trial {i}/{len(trial_specs)} | rule={main_rule:16s} | {family:11s} | F1(excl)={metrics_base['f1']:.4f} | F1(all)={metrics_all['f1']:.4f}")

        except Exception as e:
            rows.append({"trial": i, "main_rule": main_rule, "family": family, "error": str(e), "spec": spec})
            print(f"[T+{horizon}] Trial {i}/{len(trial_specs)} | rule={main_rule:16s} | {family:11s} | ERROR: {e}")

    result_df = pd.DataFrame(rows)
    return {"horizon": horizon, "result_df": result_df, "best": best}


In [ ]:
# smoke test (quick): run one lightweight trial for T+3
_old_max_trials = MAX_TRIALS_PER_HORIZON
_old_rules = MAIN_RULE_GRID.copy()
MAX_TRIALS_PER_HORIZON = 1
MAIN_RULE_GRID = ["adaptive"]

exp_smoke = run_horizon_experiment(3)
print(exp_smoke['result_df'][['trial', 'main_rule', 'family', 'f1']].head(3))

MAX_TRIALS_PER_HORIZON = _old_max_trials
MAIN_RULE_GRID = _old_rules

[T+3] Rule: train uses non-excluded only; eval reports both non-excluded and all symbols.
[T+3] Rule: signal suppressed when both volume and oi are zero.
[T+3] Feature construction: define_main_adaptive_robust + make_refined_features (from main.py).
[T+3] Feature rule: lambdarank includes symbol_cat; other models exclude symbol_cat; vshare model excludes v_share.
[T+3] Main-rule grid: ['adaptive']
[T+3] Trial 1/39 | rule=adaptive         | lgbm_rank   | F1(excl)=0.7257 | F1(all)=0.6155
[T+3] Trial 2/39 | rule=adaptive         | lgbm_rank   | F1(excl)=0.7199 | F1(all)=0.6141
[T+3] Trial 3/39 | rule=adaptive         | lgbm_rank   | F1(excl)=0.7132 | F1(all)=0.6197
[T+3] Trial 4/39 | rule=adaptive         | lgbm_rank   | F1(excl)=0.7213 | F1(all)=0.6186
[T+3] Trial 5/39 | rule=adaptive         | lgbm_rank   | F1(excl)=0.7067 | F1(all)=0.6069
[T+3] Trial 6/39 | rule=adaptive         | lgbm_rank   | F1(excl)=0.7521 | F1(all)=0.6254
[T+3] Trial 7/39 | rule=adaptive         | lgbm_rank   | F1

In [ ]:
# T+5 experiment cell
exp_t5 = run_horizon_experiment(5)
res_t5 = exp_t5['result_df'].copy()

print('=' * 80)
print('[T+5 Grid Search Finished]')
print('successful trials:', int(res_t5['f1'].notna().sum()) if 'f1' in res_t5.columns else 0)
print('=' * 80)

best_t5 = exp_t5['best']
if best_t5 is None:
    raise RuntimeError('T+5 没有成功 trial。')

print('[T+5 Best Family]', best_t5['family'])
print('[T+5 Best Main Rule]', best_t5['main_rule'])
print('[T+5 Excluded] F1={:.4f}, P={:.4f}, R={:.4f}'.format(best_t5['metrics_excluded']['f1'], best_t5['metrics_excluded']['precision'], best_t5['metrics_excluded']['recall']))
print('[T+5 ALL]      F1={:.4f}, P={:.4f}, R={:.4f}'.format(best_t5['metrics_all']['f1'], best_t5['metrics_all']['precision'], best_t5['metrics_all']['recall']))
for d in range(1, 6):
    print('  Day{} hit(excl)={:.4f} (n={})'.format(d, best_t5['metrics_excluded'][f'day{d}_hit_rate'], best_t5['metrics_excluded'][f'day{d}_count']))

display(res_t5.sort_values('f1', ascending=False).head(20)[['trial', 'main_rule', 'family', 'f1', 'precision', 'recall', 'd1_hit', 'd2_hit', 'd3_hit', 'd4_hit', 'd5_hit', 'all_f1', 'all_precision', 'all_recall']])

[T+5] Rule: train uses non-excluded only; eval reports both non-excluded and all symbols.
[T+5] Rule: signal suppressed when both volume and oi are zero.
[T+5] Feature construction: define_main_adaptive_robust + make_refined_features (from main.py).
[T+5] Feature rule: lambdarank includes symbol_cat; other models exclude symbol_cat; vshare model excludes v_share.
[T+5] Trial 1/40 | lgbm_rank   | F1(excl)=0.7346 | F1(all)=0.6523
[T+5] Trial 2/40 | lgbm_rank   | F1(excl)=0.7278 | F1(all)=0.6495
[T+5] Trial 3/40 | lgbm_rank   | F1(excl)=0.7353 | F1(all)=0.6597
[T+5] Trial 4/40 | lgbm_rank   | F1(excl)=0.7782 | F1(all)=0.6845
[T+5] Trial 5/40 | lgbm_rank   | F1(excl)=0.7744 | F1(all)=0.6826
[T+5] Trial 6/40 | lgbm_rank   | F1(excl)=0.7857 | F1(all)=0.6912
[T+5] Trial 7/40 | mlp_rank    | F1(excl)=0.7010 | F1(all)=0.6077
[T+5] Trial 8/40 | mlp_rank    | F1(excl)=0.6532 | F1(all)=0.5727
[T+5] Trial 9/40 | mlp_rank    | F1(excl)=0.7253 | F1(all)=0.6111
[T+5] Trial 10/40 | mlp_rank    | F1(exc

,trial,family,f1,precision,recall,d1_hit,d2_hit,d3_hit,d4_hit,d5_hit,all_f1,all_precision,all_recall
21,22,three_model,0.791452,0.814871,0.769341,0.533784,0.662069,0.823529,0.903704,0.954198,0.676751,0.711425,0.645299
18,19,three_model,0.790560,0.814590,0.767908,0.533784,0.662069,0.816176,0.903704,0.954198,0.678652,0.715640,0.645299
33,34,three_model,0.789550,0.800000,0.779370,0.567568,0.675862,0.823529,0.896296,0.961832,0.677348,0.701373,0.654915
30,31,three_model,0.788671,0.799705,0.777937,0.560811,0.675862,0.823529,0.896296,0.961832,0.677348,0.701373,0.654915
36,37,three_model,0.788630,0.802671,0.775072,0.560811,0.655172,0.823529,0.903704,0.961832,0.677366,0.702641,0.653846
20,21,three_model,0.787482,0.800296,0.775072,0.527027,0.689655,0.816176,0.911111,0.961832,0.695749,0.730047,0.664530
39,40,three_model,0.787172,0.801187,0.773639,0.560811,0.648276,0.823529,0.903704,0.961832,0.674792,0.700806,0.650641
29,30,three_model,0.786647,0.797059,0.776504,0.533784,0.682759,0.823529,0.911111,0.961832,0.696767,0.728438,0.667735
26,27,three_model,0.786026,0.798817,0.773639,0.527027,0.675862,0.823529,0.911111,0.961832,0.693080,0.725467,0.663462
14,15,vshare_reg,0.785822,0.829618,0.746418,0.486486,0.641379,0.808824,0.874074,0.954198,0.655329,0.698068,0.617521


In [ ]:
# T+3 experiment cell
exp_t3 = run_horizon_experiment(3)
res_t3 = exp_t3['result_df'].copy()

print('=' * 80)
print('[T+3 Grid Search Finished]')
print('successful trials:', int(res_t3['f1'].notna().sum()) if 'f1' in res_t3.columns else 0)
print('=' * 80)

best_t3 = exp_t3['best']
if best_t3 is None:
    raise RuntimeError('T+3 没有成功 trial。')

print('[T+3 Best Family]', best_t3['family'])
print('[T+3 Best Main Rule]', best_t3['main_rule'])
print('[T+3 Excluded] F1={:.4f}, P={:.4f}, R={:.4f}'.format(best_t3['metrics_excluded']['f1'], best_t3['metrics_excluded']['precision'], best_t3['metrics_excluded']['recall']))
print('[T+3 ALL]      F1={:.4f}, P={:.4f}, R={:.4f}'.format(best_t3['metrics_all']['f1'], best_t3['metrics_all']['precision'], best_t3['metrics_all']['recall']))
for d in range(1, 4):
    print('  Day{} hit(excl)={:.4f} (n={})'.format(d, best_t3['metrics_excluded'][f'day{d}_hit_rate'], best_t3['metrics_excluded'][f'day{d}_count']))

display(res_t3.sort_values('f1', ascending=False).head(20)[['trial', 'main_rule', 'family', 'f1', 'precision', 'recall', 'd1_hit', 'd2_hit', 'd3_hit', 'all_f1', 'all_precision', 'all_recall']])

[T+3] Rule: train uses non-excluded only; eval reports both non-excluded and all symbols.
[T+3] Rule: signal suppressed when both volume and oi are zero.
[T+3] Feature construction: define_main_adaptive_robust + make_refined_features (from main.py).
[T+3] Feature rule: lambdarank includes symbol_cat; other models exclude symbol_cat; vshare model excludes v_share.
[T+3] Trial 1/40 | lgbm_rank   | F1(excl)=0.7521 | F1(all)=0.6254
[T+3] Trial 2/40 | lgbm_rank   | F1(excl)=0.7572 | F1(all)=0.6271
[T+3] Trial 3/40 | lgbm_rank   | F1(excl)=0.7521 | F1(all)=0.6214
[T+3] Trial 4/40 | lgbm_rank   | F1(excl)=0.7713 | F1(all)=0.6340
[T+3] Trial 5/40 | lgbm_rank   | F1(excl)=0.7591 | F1(all)=0.6284
[T+3] Trial 6/40 | lgbm_rank   | F1(excl)=0.7698 | F1(all)=0.6397
[T+3] Trial 7/40 | mlp_rank    | F1(excl)=0.6538 | F1(all)=0.5207
[T+3] Trial 8/40 | mlp_rank    | F1(excl)=0.6032 | F1(all)=0.4877
[T+3] Trial 9/40 | mlp_rank    | F1(excl)=0.6861 | F1(all)=0.5393
[T+3] Trial 10/40 | mlp_rank    | F1(exc

In [ ]:
# Final selection summary + persist for main/backup usage
summary_rows = [
    {
        'horizon': 5,
        'best_main_rule': best_t5['main_rule'],
        'best_family': best_t5['family'],
        'best_f1_excluded': best_t5['metrics_excluded']['f1'],
        'best_precision_excluded': best_t5['metrics_excluded']['precision'],
        'best_recall_excluded': best_t5['metrics_excluded']['recall'],
        'best_f1_all': best_t5['metrics_all']['f1'],
        'best_spec': best_t5['spec'],
    },
    {
        'horizon': 3,
        'best_main_rule': best_t3['main_rule'],
        'best_family': best_t3['family'],
        'best_f1_excluded': best_t3['metrics_excluded']['f1'],
        'best_precision_excluded': best_t3['metrics_excluded']['precision'],
        'best_recall_excluded': best_t3['metrics_excluded']['recall'],
        'best_f1_all': best_t3['metrics_all']['f1'],
        'best_spec': best_t3['spec'],
    },
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df[['horizon', 'best_main_rule', 'best_family', 'best_f1_excluded', 'best_precision_excluded', 'best_recall_excluded', 'best_f1_all']])

summary_df.to_json('output_results/final_summary_best_methods.json', orient='records', force_ascii=False, indent=2)
res_t5.to_parquet('output_results/grid_results_t5.parquet', index=False)
res_t3.to_parquet('output_results/grid_results_t3.parquet', index=False)
print('Saved: output_results/final_summary_best_methods.json')
print('Saved: output_results/grid_results_t5.parquet')
print('Saved: output_results/grid_results_t3.parquet')

NameError: name 'best_t5' is not defined